<img src="https://raw.githubusercontent.com/mjcowley/astrolabs/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics" width="350"/>

# AstroLab C8: Running the Clock Backwards

**PVB220 Cosmology** | Dr Michael Cowley | michael.cowley@qut.edu.au

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjcowley/astrolabs/blob/main/AstroLabC8.ipynb)

## Introduction

In the lecture we followed the Universe from the Planck epoch to the first stars. But how do we *know* any of this happened? Nobody was there with a thermometer.

The answer is that the Universe has left us clues. The most important is that **space is expanding**, and as it expands, light travelling through it gets stretched to longer, redder wavelengths. If we measure how much light has been stretched, we can work out how big the Universe was, how hot it was, and how old it was when that light set off. In other words, we can run the clock backwards.

In this lab you'll:

1. Warm up with the **Planck units** from the lecture
2. Stretch space with the **scale factor** and see how it creates **redshift**
3. Fit the **real spectrum of the cosmic microwave background** and find out why it glows in microwaves
4. Use a **cosmic thermometer** to travel through the epochs of the Big Bang
5. Measure the **redshifts of real galaxies** observed by the James Webb Space Telescope
6. Take a sneak peek at the CMB sky as seen by three generations of space telescopes

Have Canvas open alongside this notebook and answer each 🎯 **Canvas checkpoint** as you go. If you need a reminder of how the workbooks work, see Part 0 of AstroLab C7.

▶️ Run the cell below first.

In [ ]:
#@title ▶️ Run me first: setup
import os
import urllib.request

# Download the latest helper file (keep the existing copy if we're offline)
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mjcowley/astrolabs/main/astrolabs.py",
        "astrolabs.py")
except Exception:
    if not os.path.exists("astrolabs.py"):
        raise

import numpy as np                 # numbers and arrays
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # plotting
import astropy.units as u          # physical units
from astropy.constants import c, G, hbar, h, k_B   # physical constants
import importlib
import astrolabs as al             # helper functions for these labs
importlib.reload(al)               # use the version just downloaded

al.setup()

---
# Part 1: Warm-up: The Planck Units

The lecture started at the very beginning, in the **Planck epoch**, when the Universe was so small and hot that our current laws of physics break down. We worked out three **Planck units** by hand:

$$\ell_P = \sqrt{\frac{\hbar G}{c^3}} \qquad t_P = \frac{\ell_P}{c} \qquad T_P = \sqrt{\frac{\hbar c^5}{G k^2}}$$

Doing this by hand means juggling powers of ten and cancelling units. Python can do both for us. The constants $c$, $G$, $\hbar$ and $k$ (`k_B`) come from `astropy.constants`, with their units attached.

In [ ]:
# Have a look at the constants we'll use
print("Speed of light:      ", c)
print("Gravitational const.:", G)
print("Reduced Planck const:", hbar)
print("Boltzmann constant:  ", k_B)

Notice that every constant carries its units. Watch what happens when we put them into the formula for the Planck length: Python works out the units for us.

In [ ]:
planck_length = np.sqrt(hbar * G / c**3)

print("Planck length:", planck_length)
print("In metres:    ", planck_length.to(u.m))

The first line looks strange because Python kept every unit exactly as it came out of the formula. Adding `.to(u.m)` simplifies them to metres: Python multiplied, divided and cancelled all the units, exactly as we did on the slides by crossing out units. Now the Planck time is just the Planck length divided by the speed of light.

In [ ]:
planck_time = planck_length / c

print("Planck time:", planck_time.to(u.s))

> 🎯 **Canvas checkpoint Q1:** Run the cell below to calculate the **Planck temperature**. What value do you get, in kelvin?

In [ ]:
# 👉 The Planck temperature formula is written for you. Run the cell.
planck_temperature = np.sqrt(hbar * c**5 / (G * k_B**2))

print("Planck temperature:", planck_temperature.to(u.K))
al.check("C8_planck_temperature", planck_temperature.to(u.K))

To put that in perspective, the hottest temperature ever made on Earth (in a particle collider) is about $5 \times 10^{12}$ K. The Planck temperature is about twenty orders of magnitude hotter. And the Planck length is about twenty orders of magnitude smaller than a proton!

> 🔍 **Peek inside:** if you're curious, try `planck_length / (1e-15 * u.m)` to compare the Planck length with the size of a proton, about $10^{-15}$ m.

---
# Part 2: Stretching Space

The lecture's balloon analogy is the key to everything in this lab. As the balloon inflates, the dots on its surface move apart, not because they're moving across the balloon, but because the rubber between them is stretching. Space works the same way.

Cosmologists describe the size of the Universe with the **scale factor** $a$. Today $a = 1$. When the Universe was half its current size, $a = 0.5$.

Light travelling through expanding space gets stretched along with it. The amount of stretching is the **redshift** $z$:

$$1 + z = \frac{1}{a}$$

▶️ Run the cell below and move the slider. On the left is a fixed patch of space; on the right is a light wave that set off when the Universe had scale factor $a$, and the same wave arriving at our telescopes today.

In [ ]:
al.expansion_explorer()

Notice that as you go back in time (smaller $a$), the patch of space gets **more crowded**, but it's never empty around the edges. As the lecture explained, there was no single point that exploded outwards. Space was always full of matter; it was just packed more tightly.

Let's redo the lecture's worked example in Python: when the Universe was one tenth of its present size, what redshift would we measure?

In [ ]:
# 👉 Change the scale factor and re-run
a = 0.1

z = 1 / a - 1
print(f"Scale factor a = {a}  →  redshift z = {z:.2f}")

> 🎯 **Canvas checkpoint Q2:** Light from a distant galaxy set off when the Universe was **25%** of its current size. What redshift do we measure?

---
# Part 3: Why Microwaves?

The cosmic microwave background (CMB) is light released at **recombination**, about 380,000 years after the Big Bang, when the Universe was about 3,000 K. Something that hot glows orange-red, like a stove element. So why do we see it as microwaves today?

Let's find out using real data from NASA's **COBE** satellite. Its FIRAS instrument measured the brightness of the CMB at 43 different frequencies in 1989 and 1990.

In [ ]:
firas = pd.read_csv(al.get_data("C8/firas_cmb_spectrum.csv"))

firas.head()

The columns are the frequency (GHz), the equivalent wavelength (mm), the measured brightness or **intensity** (in MJy/sr, a unit radio astronomers use), and its uncertainty.

## Planck's law

Anything that absorbs and emits light perfectly is called a **blackbody**, and its spectrum depends only on its temperature. Max Planck (yes, the same Planck!) worked out the formula in 1900:

$$B_\nu(T) = \frac{2 h \nu^3}{c^2} \, \frac{1}{e^{h\nu / kT} - 1}$$

Here it is as a Python **function**: give it frequencies and a temperature, and it gives back the predicted intensity.

In [ ]:
def planck_intensity(frequency_ghz, temperature):
    # Planck's law for a blackbody, returning intensity in MJy/sr
    nu = frequency_ghz * 1e9                                   # GHz to Hz
    x = h.value * nu / (k_B.value * temperature)
    intensity = 2 * h.value * nu**3 / c.value**2 / np.expm1(x)  # W / m^2 / Hz / sr
    return intensity * 1e20                                    # to MJy/sr

# Try it: the predicted intensity at 200 GHz for a 2.7 K blackbody
print(planck_intensity(200, 2.7), "MJy/sr")

## Fit the temperature of the Universe

▶️ Run the cell below and move the temperature slider until the red curve passes through the blue data points. Watch the "mismatch" number: the smaller it is, the better the fit.

In [ ]:
al.blackbody_fitter(firas["frequency_ghz"], firas["intensity_mjy_sr"],
                    firas["error_mjy_sr"], planck_intensity)

The error bars are drawn **100 times bigger** than they really are, otherwise you couldn't see them at all. This is the most perfect blackbody spectrum ever measured in nature. John Mather won the 2006 Nobel Prize in Physics for this measurement.

Fitting by eye gets you close. To find the very best temperature, we can ask the computer to do it with `curve_fit` from the `scipy` package. It tries different temperatures until the mismatch is as small as possible.

In [ ]:
from scipy.optimize import curve_fit

best, covariance = curve_fit(planck_intensity, firas["frequency_ghz"],
                             firas["intensity_mjy_sr"], p0=[3.0],
                             sigma=firas["error_mjy_sr"], absolute_sigma=True)

best_temperature = best[0]
uncertainty = np.sqrt(covariance[0, 0])
print(f"Best-fit CMB temperature: {best_temperature:.5f} ± {uncertainty:.5f} K")

> 🎯 **Canvas checkpoint Q3:** What is the temperature of the cosmic microwave background, to three decimal places?

## Where does it peak?

**Wien's law** tells us the wavelength where a blackbody is brightest: $\lambda_{\rm peak} = \dfrac{2.898 \text{ mm K}}{T}$.

In [ ]:
peak_wavelength = 2.898 * u.mm * u.K / (best_temperature * u.K)

print("The CMB is brightest at:", peak_wavelength.to(u.mm))

About a millimetre: that's the **microwave** part of the spectrum, the same kind of light a microwave oven uses (which works at about 12 cm).

💡 You might notice the FIRAS data peaks at about 160 GHz, which is a wavelength of about 1.9 mm, not 1.06 mm. Both are right! The peak lands in a different place depending on whether you measure brightness per unit of frequency (as FIRAS did) or per unit of wavelength (as Wien's law does). From here on we'll work in wavelength.

> 🎯 **Canvas checkpoint Q4:** At what wavelength is the CMB brightest today?

## Rewind the clock

The CMB was released at about 3,000 K. Since then, the expansion of space has stretched its wavelengths, and **stretching a blackbody spectrum cools it**. The temperature falls in exactly the same way as the wavelengths grow:

$$T = T_0\,(1 + z)$$

where $T_0 = 2.725$ K is the temperature today. This is the lecture's "Deeper Dive" question: how much has the CMB been redshifted since decoupling?

In [ ]:
temperature_at_decoupling = 3000    # K, from the lecture
temperature_today = 2.725           # K, from your fit

z_decoupling = temperature_at_decoupling / temperature_today - 1
print(f"Redshift of the CMB since decoupling: z = {z_decoupling:.0f}")
print(f"The Universe was 1/{1 + z_decoupling:.0f} of its present size")

> 🎯 **Canvas checkpoint Q5:** By roughly how much has the light from the CMB been redshifted since decoupling?

Now let's watch it happen. ▶️ Run the cell below and slide the redshift from 0 (today) back to 1,100 (decoupling). The yellow band shows visible light.

In [ ]:
al.cmb_rewind(planck_intensity)

At $z \approx 1{,}100$ the peak sits in the **infrared**, just beyond red light, with the short-wavelength side spilling into the visible band. If you could have stood in the Universe at recombination, the whole sky would have glowed a deep orange, in every direction. The expansion of space has since stretched that glow by a factor of about 1,100, all the way into microwaves.

> 🎯 **Canvas checkpoint Q6:** At $z = 1{,}100$, in which part of the spectrum is the CMB brightest?

---
# Part 4: The Cosmic Thermometer

Because $T = T_0(1 + z)$, the temperature of the CMB works like a thermometer for the whole Universe. Combine it with a model of how fast the Universe has expanded, and we can work out its **age** at any temperature. That gives us the timeline from the lecture.

▶️ Run the cell below. The slider sets the age of the Universe, from 100 seconds to today. The shaded bands are the epochs from the lecture.

In [ ]:
al.time_machine()

> 🔍 **Peek inside:** the ages come from `age_of_universe` in `astrolabs.py`, which adds up how long the Universe took to expand by each tiny step, using measurements from the Planck satellite.

> 🎯 **Canvas checkpoint Q7:** Use the time machine to find roughly how old the Universe was when its temperature fell to 3,000 K and the CMB was released.

## Stop 1: The first three minutes

Early on, when the Universe was about 100 seconds old and around a billion kelvin, protons and neutrons fused together. Almost all of the neutrons ended up in **helium**, with leftover protons staying as **hydrogen**.

By the time fusion started, there was about **one neutron for every seven protons**. Since each helium nucleus holds two neutrons and two protons, the fraction of mass that ends up as helium is:

$$Y = \frac{2x}{1 + x}$$

where $x$ is the number of neutrons per proton. ▶️ Move the slider to $x = 1/7 \approx 0.143$.

In [ ]:
al.helium_explorer()

> 🎯 **Canvas checkpoint Q8:** If there was one neutron for every seven protons, what fraction of the mass of the early Universe ended up as helium?

That's why the first stars (the lecture's **Population III** stars) were made of about three-quarters hydrogen and one-quarter helium, with no heavier elements at all. Every other element, including the carbon in your body, was made later inside stars.

---
# Part 5: Measuring Cosmological Redshifts with JWST

Redshift is how we measure the distances and ages of everything in the distant Universe, and you'll use it again in later labs. Let's measure some for real.

These spectra were taken by the **NIRSpec** instrument on the James Webb Space Telescope as part of **JADES**, the JWST Advanced Deep Extragalactic Survey, which has found some of the most distant galaxies known. A **spectrum** spreads a galaxy's light out by wavelength, like a rainbow.

In [ ]:
spectra = pd.read_csv(al.get_data("C8/jwst_spectra.csv"))

print("Galaxies:", list(spectra["galaxy"].unique()))
spectra.head()

Let's plot the spectrum of galaxy A. We **select** just the rows for galaxy A, then plot flux against wavelength.

In [ ]:
# 👉 Change "A" to another galaxy (B, C, D or E) and re-run
galaxy = "A"

one = spectra[spectra["galaxy"] == galaxy]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(one["wavelength_um"], one["flux"], color="k", lw=1)
ax.set_xlabel("Observed wavelength (micrometres)")
ax.set_ylabel("Flux")
ax.set_title(f"JWST spectrum of galaxy {galaxy}")
plt.show()

The sharp spikes are **emission lines**: light given out by hot gas around young stars, at very specific wavelengths set by atomic physics. Here they are, as measured in a laboratory on Earth (their **rest wavelengths**):

| Line | Element | Rest wavelength (μm) |
| --- | --- | :-: |
| Lyα (Lyman-alpha) | Hydrogen | 0.1216 |
| [O II] | Oxygen | 0.3727 |
| Hβ (H-beta) | Hydrogen | 0.4861 |
| [O III] | Oxygen | 0.5007 |
| Hα (H-alpha) | Hydrogen | 0.6563 |

In the galaxy spectra, these lines appear at **longer** wavelengths, stretched by the expansion of space:

$$\lambda_{\rm observed} = \lambda_{\rm rest}\,(1 + z)$$

## Find the redshift

▶️ Run the cell below. Pick a galaxy, then move the redshift slider until the red dashed lines sit on top of the emission lines. The bright pair of [O III] and Hβ, with Hα further along, is usually the easiest pattern to spot.

💡 You can click on the number next to the slider and type a value to fine-tune it.

In [ ]:
al.redshift_explorer(spectra)

Write down the redshift you measure for each galaxy, then enter them in the cell below.

> 🎯 **Canvas checkpoint Q9:** What is the redshift of **galaxy C** (to two decimal places)?

In [ ]:
# 👉 Replace each None with the redshift you measured
my_redshifts = {
    "A": None,
    "B": None,
    "C": None,
    "D": None,
    "E": None,
}

al.check("C8_z_galaxy_C", my_redshifts["C"])

## What your redshifts tell you

Now let's run the clock backwards for each galaxy. For each redshift, the cell below works out how big the Universe was, how hot the CMB was, how old the Universe was when the light set off, and how long the light has been travelling.

In [ ]:
rows = []
for galaxy, z in my_redshifts.items():
    if z is None:
        continue          # skip galaxies you haven't measured yet
    age_then = al.age_of_universe(z)
    rows.append({
        "galaxy": galaxy,
        "redshift z": z,
        "size vs today": f"{1 / (1 + z):.0%}",
        "CMB temperature (K)": round(al.T_CMB_TODAY * (1 + z), 1),
        "age then (billion yr)": round(age_then.to(u.Gyr).value, 2),
        "light travel time (billion yr)": round(13.80 - age_then.to(u.Gyr).value, 2),
    })

results = pd.DataFrame(rows)
results

> 🎯 **Canvas checkpoint Q10:** How big was the Universe, compared with today, when the light left **galaxy E**?

## The Lyman break

Look again at galaxies C, D and E. Below a certain wavelength, their light drops to almost **zero**. That edge sits at the redshifted wavelength of Lyman-alpha, $0.1216\,(1+z)$ μm.

Light bluer than Lyman-alpha has enough energy to be absorbed by **neutral hydrogen atoms**. In the early Universe, during and just after the **Dark Ages**, intergalactic space was full of neutral hydrogen left over from recombination, which soaked up that light before it could reach us. This "Lyman break" is one of the main ways astronomers find the most distant galaxies, including MoM-z14 from Lecture 1. Mapping how it fades away tells us when the first stars and galaxies **reionised** the Universe.

> 🎯 **Canvas checkpoint Q11:** Why does the light from the most distant galaxies drop to almost zero at short wavelengths?

---
# Part 6: Sneak Peek: The Baby Picture of the Universe

The CMB is a snapshot of the Universe at 380,000 years old. Let's look at it the way astronomers do, using real maps from the WMAP and Planck satellites.

In [ ]:
from astropy.io import fits

cmb_file = fits.open(al.get_data("C8/cmb_maps.fits"))
cmb_maps = {hdu.name: hdu.data for hdu in cmb_file[1:]}
print("Maps in the file:", list(cmb_maps))

▶️ Run the cell below and click through the layers, from 1 to 4. Each map shows the whole sky, with the Milky Way running across the middle.

In [ ]:
al.cmb_layers_explorer(cmb_maps)

1. At first glance the CMB is **perfectly smooth**: 2.725 K in every direction. This is the evidence for the cosmological principle from Lecture 1.
2. Zoom in by a factor of 1,000 and one half of the sky is slightly hotter. That's because we're moving through space at about 370 km/s, so the CMB ahead of us is blueshifted (hotter) and behind us is redshifted (cooler). It's redshift again, this time from our own motion!
3. Remove our motion and the Milky Way appears, glowing in microwaves.
4. Remove the Milky Way and we finally see the **ripples**: differences of just one part in 100,000. These tiny lumpier regions grew, under gravity, into every galaxy in the Universe.

> 🎯 **Canvas checkpoint Q12:** In layer 2, what causes one half of the sky to look hotter than the other?

## Sharper and sharper

▶️ Finally, here's the same 10° × 10° patch of sky as it would look to three generations of CMB satellites. Click through them.

In [ ]:
al.resolution_explorer(cmb_maps)

COBE first detected the ripples in 1992, but could only see them as faint smudges. WMAP and Planck revealed them in sharper and sharper detail. You'll see in Week 9 how the sizes of these ripples tell us what the Universe is made of.

---
# What's Coming Up

| Lab | Week | What you'll do |
| --- | :-: | --- |
| C7 | 2 | Python toolkit |
| **C8** | 4 | The Big Bang and the CMB (this lab!) |
| **C9** | 6 | Galaxy colours, star formation and active galactic nuclei |
| **C10** | 8 | Gravitational waves: listen to colliding black holes |
| **C11** | 10 | Cosmology surveys: cross-matching catalogues and scaling relations |

### Sparkler ideas from this lab

* The CMB is the most perfect blackbody in nature: what does that tell us?
* Rewinding the CMB: from an orange glow to microwaves
* Measuring the redshifts of JWST galaxies and the Lyman break
* From COBE to Planck: how the baby picture of the Universe came into focus

**Well done, you've finished AstroLab C8!** Make sure you've answered all the 🎯 Canvas checkpoints.

---
# 🚀 Stretch Activity: Faster Than Light?

*Optional, if you have time left.*

In the 1920s, Edwin Hubble turned redshifts into speeds using the Doppler formula $v = cz$. Let's try that on the galaxies you measured.

In [ ]:
for galaxy, z in my_redshifts.items():
    if z is None:
        continue
    v = c * z
    print(f"Galaxy {galaxy}: z = {z:.2f}  →  v = cz = {v.to(u.km / u.s):,.0f} ({z:.1f} times the speed of light!)")

Galaxies moving many times faster than light? That can't be right, can it?

The formula $v = cz$ only works for small redshifts. Cosmological redshift isn't really caused by galaxies moving *through* space: it's caused by space itself stretching while the light travels, which is why we used $1 + z = 1/a$ instead.

We can still ask how fast the distance to each galaxy is growing *today*. That's the Hubble constant times the galaxy's distance now:

In [ ]:
from astropy.cosmology import Planck18 as cosmo

for galaxy, z in my_redshifts.items():
    if z is None:
        continue
    distance_now = cosmo.comoving_distance(z)
    speed_now = cosmo.H0 * distance_now
    print(f"Galaxy {galaxy}: distance now = {distance_now.to(u.lyr).value / 1e9:.1f} billion light-years, "
          f"growing at {(speed_now / c).decompose():.1f} times the speed of light")

The distances to all of these galaxies are growing **faster than light** today, and that's perfectly fine! As the lecture on inflation explained, nothing can travel *through* space faster than light, but space itself can stretch as fast as it likes.

🚀 **Extra challenge:** galaxy E's light has been travelling for about 13.3 billion years, yet the galaxy is now about 31 billion light-years away. Can you explain how both can be true? (Hint: think about what happened to the space the light had already crossed.)

---
### Data and image credits

* COBE FIRAS CMB spectrum: Fixsen et al. 1996, ApJ, 473, 576, via NASA LAMBDA
* JWST NIRSpec spectra: JADES Data Release 4 (Curtis-Lake et al. 2025; Scholtz, Carniani et al. 2025), via MAST (doi:10.17909/8tdj-8n28)
* Planck CMB map: Planck Collaboration 2016, A&A, 594, A9 (SMICA), via NASA/IPAC IRSA
* WMAP K-band map: Bennett et al. 2013, ApJS, 208, 20, via NASA LAMBDA
* CMB dipole: Planck Collaboration 2020, A&A, 641, A1
* Cosmological parameters: Planck Collaboration 2020, A&A, 641, A6